In [ ]:
import torch

__all__ = ["PreFetcher"]


class PreFetcher:
    r""" Data pre-fetcher to accelerate the data loading
    """

    # 初始方法
    def __init__(self, loader, ismulti):
        # 原始数据加载器
        self.ori_loader = loader
        # 数据批次数量
        self.len = len(loader)
        # 创建专用CUDA流，用于数据预取
        self.stream = torch.cuda.Stream()
        # 初始化预取数据存储，用于存储预取的下一个批次数据
        self.next_input = None
        # 多模态标志
        self.ismulti = ismulti

    # 数据预取方法
    def preload(self):
        try:
            # 获取下一批数据
            self.next_input = next(self.loader)
        except StopIteration:
            self.next_input = None
            return

        # 在预取流中将数据异步(非阻塞)传输到GPU
        with torch.cuda.stream(self.stream):
            for idx, tensor in enumerate(self.next_input):
                self.next_input[idx] = tensor.cuda(non_blocking=True)

    # 返回数据长度
    def __len__(self):
        return self.len

    # 迭代器初始化
    def __iter__(self):
        # step1：for循环调用__iter__，将原始数据加载器转换为迭代器
        self.loader = iter(self.ori_loader)

        # step2：从从磁盘加载 Batch1 到 CPU -> 在预取流中将 Batch1 异步传输到GPU -> 存储到 self.next_input
        self.preload()
        return self

    # 获取下一批数据
    def __next__(self):
        # step3、4、5……
        # ① 主计算流等待预取流完成 Batch1 传输
        torch.cuda.current_stream().wait_stream(self.stream)
        # ② 将 Batch1 从 next_input 取出返回
        input = self.next_input

        # 迭代终止条件
        if input is None:
            raise StopIteration

        # ③记录张量使用的流
        for idx, tensor in enumerate(input):
            tensor.record_stream(torch.cuda.current_stream())
        # ④立即开始预取下一批：从磁盘加载 Batch2 到 CPU -> 在预取流中传输 Batch2 到 GPU -> 存储到 self.next_input
        self.preload()

        # 返回当前批次
        return input

In [2]:
# 展示dataloader是可迭代对象，而不是迭代器
import torch
from torch.utils.data import DataLoader, TensorDataset

# 创建数据集
data = torch.arange(10)
dataset = TensorDataset(data)
dataloader = DataLoader(dataset, batch_size=2)

# 第一次遍历
print("第一次遍历:")
for batch in dataloader:
    print(batch)

# 第二次遍历（不会输出任何内容）
print("\n第二次遍历:")
for batch in dataloader:
    print(batch)  # 不会打印任何内容，因为迭代器已耗尽

第一次遍历:
[tensor([0, 1])]
[tensor([2, 3])]
[tensor([4, 5])]
[tensor([6, 7])]
[tensor([8, 9])]

第二次遍历:
[tensor([0, 1])]
[tensor([2, 3])]
[tensor([4, 5])]
[tensor([6, 7])]
[tensor([8, 9])]
